# VK Advertising — Predicting User Gender

**Task (VK / All Cups).** A company selling holiday gift sets wants gender-targeted promotions on
VKontakte, Odnoklassniki and Zen. We build a model that predicts a user's **gender**
(`target` ∈ {0,1}) from their ad-request logs, so ads reach the right audience.

**This notebook is self-contained and idempotent** — *Restart Kernel → Run All* reproduces everything
and can be re-run top-to-bottom safely. It produces three deliverables:

| Output | File | Format |
|---|---|---|
| Predictions | `submission.csv` | `user_id;target` (same as `train_labels.csv`), one row per `test_users` user |
| Trained model | `model.pkl` | 5-fold LightGBM ensemble + metadata (`pickle`) |
| Report | `report.pdf` | project description, method, results |

### Data (all use `;` separator)
| File | Content |
|---|---|
| `train.csv` / `test.csv` | request logs: `request_ts, user_id, referer, geo_id, user_agent` |
| `train_labels.csv` | `user_id, target` (gender) |
| `test_users.csv` | the users to predict |
| `referer_vectors.csv` | 10-dim embedding `component0..9` per `referer` |
| `geo_info.csv` | *optional* — `geo_id, country_id, region_id, timezone` (auto-used if present) |

### Approach
Users have only ~1.14 requests each (median 1), so the signal is essentially **per-request**.
We turn each request into features — the **referer embedding** (strongest signal: encodes the *type*
of site the ad ran on), **user-agent** (browser/OS/device generation), **geo**, **time-of-day** — and
aggregate to the user. High-cardinality categoricals (`domain`, `geo_id`, `browser`, …) use
**out-of-fold smoothed target encoding** (no leakage). A **LightGBM** classifier is trained with
5-fold cross-validation.


## 1. Dependencies
Only standard scientific libraries are needed (`pickle` is built-in). Uncomment to install on a fresh machine.

In [ ]:
# %pip install -q pandas numpy scikit-learn lightgbm matplotlib reportlab
import sys; print('Python', sys.version.split()[0])

In [ ]:
import os, re, ast, time, warnings, pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

SEP, SEED, N_FOLDS, SMOOTH = ';', 42, 5, 20
DATA_DIR = '.'                                  # folder with the CSV files
VEC = [f'component{i}' for i in range(10)]      # referer embedding columns
np.random.seed(SEED)
_t0 = time.time()
def log(*a): print(f'[{time.time()-_t0:6.1f}s]', *a)

## 2. Load data (raw, never mutated)

In [ ]:
train_raw  = pd.read_csv(f'{DATA_DIR}/train.csv',        sep=SEP)
test_raw   = pd.read_csv(f'{DATA_DIR}/test.csv',         sep=SEP)
labels     = pd.read_csv(f'{DATA_DIR}/train_labels.csv', sep=SEP)
test_users = pd.read_csv(f'{DATA_DIR}/test_users.csv',   sep=SEP)
rv         = pd.read_csv(f'{DATA_DIR}/referer_vectors.csv', sep=SEP).drop_duplicates('referer')

GEO_PATH = f'{DATA_DIR}/geo_info.csv'
geo = pd.read_csv(GEO_PATH, sep=SEP) if os.path.exists(GEO_PATH) else None

lab_map   = labels.set_index('user_id')['target']
lab_index = set(lab_map.index)
test_uset = set(test_users['user_id'])
log('train', train_raw.shape, '| test', test_raw.shape, '| labels', labels.shape,
    '| referer_vectors', rv.shape, '| geo', 'absent' if geo is None else geo.shape)
train_raw.head()

## 3. Quick EDA

In [ ]:
print('Target balance     :', labels.target.value_counts(normalize=True).round(3).to_dict())
print('Requests per user  :', train_raw.groupby('user_id').size().describe()[['mean','50%','max']].round(2).to_dict())
print('Unique referers    :', train_raw.referer.nunique(),
      '| domains:', train_raw.referer.str.extract(r'https?://([^/]+)')[0].nunique(),
      '| geo_id:', train_raw.geo_id.nunique())
print('Vector coverage    : train', round(train_raw.referer.isin(set(rv.referer)).mean(),4),
      '| test', round(test_raw.referer.isin(set(rv.referer)).mean(),4))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
labels.target.value_counts().sort_index().plot.bar(ax=ax[0], color=['#4C72B0','#DD8452'])
ax[0].set_title('Target (gender) balance'); ax[0].set_xlabel('target'); ax[0].set_ylabel('users')
train_raw.groupby('user_id').size().value_counts().sort_index().plot.bar(ax=ax[1], color='#55A868')
ax[1].set_title('Requests per user'); ax[1].set_xlabel('# requests'); ax[1].set_ylabel('users')
plt.tight_layout(); plt.savefig('eda_overview.png', dpi=110, bbox_inches='tight'); plt.show()

## 4. Request-level features

`build_request_features` always starts from the **raw** frame and returns a new one, so the cell is
**idempotent** (safe to re-run). No `category` dtype is created anywhere — `browser`/`os` are kept as
plain strings because they are used only *through* target encoding.


In [ ]:
def parse_ua(s):
    try: d = ast.literal_eval(s)
    except Exception: d = {}
    return (str(d.get('browser', 'NA')), str(d.get('os', 'NA')),
            d.get('browser_version', '0'), d.get('os_version', '0'))

def major(v):
    m = re.match(r'\s*(\d+)', str(v))
    return int(m.group(1)) if m else -1

def build_request_features(raw):
    df = raw.merge(rv, on='referer', how='left')          # adds component0..9
    ua = df['user_agent'].apply(parse_ua)
    df['browser']  = ua.str[0]                            # plain str (NOT category)
    df['os']       = ua.str[1]
    df['bver']     = ua.str[2].map(major).astype('float64')
    df['over']     = ua.str[3].map(major).astype('float64')
    df['domain']   = df['referer'].str.extract(r'https?://([^/]+)')[0].fillna('NA')
    df['has_path'] = (df['referer'].str.extract(r'https?://[^/]+/(.*)')[0].fillna('') != '').astype('float64')

    ts = pd.to_datetime(df['request_ts'], unit='s')
    if geo is not None and 'timezone' in geo.columns:
        off = pd.to_numeric(df['geo_id'].map(geo.set_index('geo_id')['timezone']),
                            errors='coerce').fillna(3).to_numpy()
        off = np.where(np.abs(off) > 24, off / 3600.0, off)   # seconds -> hours if large
        ts = ts + pd.to_timedelta(off, unit='h')
    df['hour'] = ts.dt.hour.astype('float64')
    df['dow']  = ts.dt.dayofweek.astype('float64')
    if geo is not None:
        for c in ['country_id', 'region_id']:
            if c in geo.columns:
                df[c] = df['geo_id'].map(geo.set_index('geo_id')[c])
    return df

tr = build_request_features(train_raw)
te = build_request_features(test_raw)
tr = tr[tr['user_id'].isin(lab_index)].copy()
tr['target'] = tr['user_id'].map(lab_map).astype(int)
log('request features built | train rows', tr.shape, '| test rows', te.shape)

## 5. Out-of-fold smoothed target encoding

Each category is replaced by its smoothed mean target, computed **out-of-fold** (folds assigned per
*user*) so a user's own label never leaks. Results are forced to `float`.

$$\text{enc}(c)=\frac{\sum y_c + \alpha\,\bar y}{n_c + \alpha},\qquad \alpha=20$$


In [ ]:
TE_COLS = ['domain', 'geo_id', 'browser', 'os', 'bver', 'over', 'hour']
if geo is not None:
    TE_COLS += [c for c in ['country_id', 'region_id'] if c in tr.columns]
GLOBAL = float(tr['target'].mean())

# assign each labelled user to a fold (stratified)
uids = labels[labels.user_id.isin(lab_index)][['user_id', 'target']].reset_index(drop=True)
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
uids['fold'] = -1
for f, (_, vi) in enumerate(skf.split(uids, uids.target)):
    uids.loc[vi, 'fold'] = f
tr['fold'] = tr['user_id'].map(uids.set_index('user_id')['fold'])

def smooth(grp):
    return (grp['sum'] + SMOOTH * GLOBAL) / (grp['count'] + SMOOTH)

for col in TE_COLS:
    oof = np.full(len(tr), GLOBAL, dtype='float64')
    for f in range(N_FOLDS):
        tri = (tr['fold'] != f).to_numpy(); vai = (tr['fold'] == f).to_numpy()
        enc = smooth(tr.loc[tri].groupby(col)['target'].agg(['sum', 'count']))
        oof[vai] = tr.loc[vai, col].map(enc).fillna(GLOBAL).to_numpy(dtype='float64')
    tr[f'te_{col}'] = oof
    enc_full = smooth(tr.groupby(col)['target'].agg(['sum', 'count']))   # full fit for test
    te[f'te_{col}'] = te[col].map(enc_full).fillna(GLOBAL).to_numpy(dtype='float64')
log('target encoding done:', TE_COLS)

## 6. Aggregate to one row per user

Only explicitly **numeric** columns are aggregated, each cast to `float64` first — so the grouping can
never hit a non-numeric dtype. Embeddings → mean (+std), encodings → mean, versions → max, plus
activity counts.


In [ ]:
TE_FEAT  = [f'te_{c}' for c in TE_COLS]
MEAN_COLS = VEC + TE_FEAT + ['hour', 'dow', 'has_path']
MAX_COLS  = ['bver', 'over']

def aggregate_users(df):
    d = df[['user_id', 'domain', 'geo_id'] + MEAN_COLS + MAX_COLS].copy()
    for c in MEAN_COLS + MAX_COLS:
        d[c] = d[c].astype('float64')                 # guarantees numeric aggregation
    gb = d.groupby('user_id')
    g = gb[MEAN_COLS].mean()
    g[MAX_COLS] = gb[MAX_COLS].max()
    g['n_req'] = gb.size()
    g['n_dom'] = gb['domain'].nunique()
    g['n_geo'] = gb['geo_id'].nunique()
    g = g.join(gb[VEC].std().add_suffix('_std').fillna(0.0))
    return g

Xtr = aggregate_users(tr)
ytr = Xtr.index.map(lab_map).astype(int).to_numpy()
Xte = aggregate_users(te)
Xte = Xte.loc[Xte.index.isin(test_uset)]              # only required users
FEATURES = list(Xtr.columns)
Xte = Xte.reindex(columns=FEATURES)
log('user matrices  Xtr', Xtr.shape, '| Xte', Xte.shape)
assert all(str(Xtr[c].dtype) != 'category' for c in FEATURES), 'unexpected category dtype'
Xtr.head()

## 7. Cross-validated LightGBM
Metric: ROC-AUC. Test prediction = mean of the 5 fold models.

In [ ]:
params = dict(objective='binary', metric='auc', boosting_type='gbdt',
              learning_rate=0.03, num_leaves=63, max_depth=-1,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              min_child_samples=100, lambda_l2=1.0, n_jobs=-1, verbose=-1, seed=SEED)

skf2 = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + 1)
oof  = np.zeros(len(Xtr)); test_pred = np.zeros(len(Xte)); models = []
for f, (ti, vi) in enumerate(skf2.split(Xtr, ytr)):
    dtr = lgb.Dataset(Xtr.iloc[ti], ytr[ti]); dva = lgb.Dataset(Xtr.iloc[vi], ytr[vi])
    m = lgb.train(params, dtr, num_boost_round=3000, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    oof[vi]    = m.predict(Xtr.iloc[vi])
    test_pred += m.predict(Xte) / N_FOLDS
    models.append(m)
    log(f'fold {f}: AUC={roc_auc_score(ytr[vi], oof[vi]):.5f}  best_iter={m.best_iteration}')

CV_AUC = roc_auc_score(ytr, oof)
CV_ACC = accuracy_score(ytr, (oof > 0.5).astype(int))
print(f'\n==>  CV ROC-AUC = {CV_AUC:.5f}   |   CV Accuracy = {CV_ACC:.5f}')

## 8. Feature importance

In [ ]:
imp = pd.Series(np.mean([m.feature_importance('gain') for m in models], axis=0),
                index=FEATURES).sort_values()
plt.figure(figsize=(7, 8)); imp.tail(25).plot.barh(color='#4C72B0')
plt.title('LightGBM feature importance (gain)'); plt.tight_layout()
plt.savefig('feature_importance.png', dpi=110, bbox_inches='tight'); plt.show()
imp.sort_values(ascending=False).head(12)

## 9. Save model & build submission
Classes are nearly balanced → threshold at 0.5. Output matches `train_labels.csv`.

In [ ]:
with open('model.pkl', 'wb') as fh:
    pickle.dump({'models': models, 'features': FEATURES, 'params': params,
                 'te_cols': TE_COLS, 'global_mean': GLOBAL,
                 'cv_auc': CV_AUC, 'cv_acc': CV_ACC}, fh)
log('model saved -> model.pkl')

sub = test_users[['user_id']].copy()
sub['target'] = sub['user_id'].map(pd.Series(test_pred, index=Xte.index)).fillna(GLOBAL)
sub['target'] = (sub['target'] > 0.5).astype(int)
sub.to_csv('submission.csv', sep=SEP, index=False)
print('submission.csv ->', sub.shape, '| rows == test_users:', len(sub) == len(test_users),
      '| predicted male-share:', round(sub.target.mean(), 4))
sub.head()

## 10. Generate PDF report

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle)

st = getSampleStyleSheet(); H, H2, P = st['Heading1'], st['Heading2'], st['BodyText']; P.spaceAfter = 6
story = []
def para(t, s=P): story.append(Paragraph(t, s))

para('VK Advertising — Predicting User Gender', H)
para('Machine-learning model report — VK / All Cups task', st['Italic']); story.append(Spacer(1, .3*cm))

para('1. Problem', H2)
para('A company selling holiday gift sets wants gender-targeted promotions on VKontakte, '
     'Odnoklassniki and Zen. We predict a user\'s gender (<b>target &isin; {0,1}</b>) from ad-request '
     'logs so each ad reaches the right audience. Quality is judged by ROC-AUC / accuracy on a hidden test set.')

para('2. Data &amp; key insight', H2)
para('Per-request logs for 500k labelled training users and 85k test users, a 10-dimensional embedding '
     'for every referer URL, and optional geo metadata. Crucially, each user has only ~1.14 requests on '
     'average (median 1), so the predictive signal is essentially <b>per-request</b>. Referer embeddings '
     'cover 100% of train and test requests.')

para('3. Features', H2)
rows = [['Group', 'Features', 'Why'],
    ['Referer embedding', 'component0..9 (mean + std)', 'Encodes the type of site the ad ran on — strongest signal'],
    ['User-agent', 'browser, os, browser/os major version', 'Device & software generation correlate with demographics'],
    ['Referer structure', 'domain (target-encoded), has_path', 'Which host; homepage vs deep link'],
    ['Geo', 'geo_id / country / region (target-encoded)', 'Regional gender patterns'],
    ['Time', 'hour, day-of-week', 'Activity rhythm differs by audience'],
    ['Activity', 'n_requests, n_domains, n_geos', 'Engagement intensity']]
t = Table(rows, colWidths=[3.2*cm, 6.0*cm, 7.0*cm])
t.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#4C72B0')),
    ('TEXTCOLOR',(0,0),(-1,0),colors.white),('FONTSIZE',(0,0),(-1,-1),8),
    ('GRID',(0,0),(-1,-1),.4,colors.grey),('VALIGN',(0,0),(-1,-1),'TOP'),
    ('ROWBACKGROUNDS',(0,1),(-1,-1),[colors.white,colors.HexColor('#EEF2F8')])]))
story.append(t); story.append(Spacer(1, .3*cm))
para('High-cardinality categoricals use <b>out-of-fold smoothed target encoding</b> (&alpha;=20) so '
     'labels never leak; request features are then aggregated to one row per user.')

para('4. Model &amp; validation', H2)
para('A <b>LightGBM</b> gradient-boosted tree (lr 0.03, 63 leaves, feature/bagging fraction 0.8, '
     'L2 = 1.0) trained with <b>5-fold stratified cross-validation</b> and early stopping. Test '
     'predictions average the five fold models; a 0.5 threshold gives the 0/1 label.')

para('5. Results', H2)
res = [['Metric', 'Value (5-fold CV)'],
       ['ROC-AUC', f'{CV_AUC:.4f}'], ['Accuracy', f'{CV_ACC:.4f}'],
       ['Predicted test male-share', f'{sub.target.mean():.3f}']]
t2 = Table(res, colWidths=[8*cm, 6*cm])
t2.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#55A868')),
    ('TEXTCOLOR',(0,0),(-1,0),colors.white),('FONTSIZE',(0,0),(-1,-1),9),
    ('GRID',(0,0),(-1,-1),.4,colors.grey)]))
story.append(t2); story.append(Spacer(1, .3*cm))
if os.path.exists('feature_importance.png'):
    para('Top predictive features:'); story.append(Image('feature_importance.png', width=11*cm, height=12*cm))

para('6. Deliverables &amp; reproducibility', H2)
para('Running this notebook end-to-end regenerates <b>model.pkl</b>, <b>submission.csv</b> '
     '(predictions for every test_users user, in train_labels format) and this <b>report.pdf</b>. '
     'The pipeline is deterministic (fixed seeds) and runs in a few minutes on a laptop CPU.')

SimpleDocTemplate('report.pdf', pagesize=A4, leftMargin=2*cm, rightMargin=2*cm,
                  topMargin=1.6*cm, bottomMargin=1.6*cm).build(story)
print('report.pdf generated')

---
### Generated files
* **`submission.csv`** — `user_id;target` for every user in `test_users.csv`
* **`model.pkl`** — 5-fold LightGBM ensemble + metadata
* **`report.pdf`** — project report

*Self-contained & idempotent: Restart Kernel → Run All reproduces all outputs.*